# VSG-SMIB MATLAB replications

## Case summary

| ID | Case | CodeSim model | Disturbance and recovery | Purpose / reference |
|---|---|---|---|---|
| 1 | Single current limiting | `VSGSMIBSystemNoCapacitor` | 0.25 pu fault at 0.2 s; fixed recovery at 0.5 s | Replicates `Single_20261005_233637` |
| 2 | Single current limiting | `VSGSMIBSystemNoCapacitor` | 0.25 pu fault at 0.2 s; angle-triggered recovery | Replicates `Single_20261005_232639` and `Single_20261003_230407` |
| 3 | Overvoltage | Archived `VSGCL` replay | Recovery at -90 degrees | Replicates `G2_angle_m90` |
| 4 | Undervoltage | Archived `VSGCL` replay | Recovery at +90 degrees | Replicates `G2_angle_90` |
| 5 | Undervoltage then overvoltage | Archived `VSGCL` replay | 80% fault depth; fixed recovery | Replicates `G4_depth_0p8` |
| 6 | Temporary overvoltage | `GFLLVRTSystem` | Default fault and recovery parameters | Default GFL LVRT case |
| 7 | Recurrent FRT | `GFLLVRTSystemCapacitor` | Recurrent threshold crossings; focused recovery window | Capacitor-based GFL case |
| 8 | Recurrent current limiting | `VSGSMIBSystem` with `VSGCC` | 0.5 pu fault at 0.2 s; recovery at 0.3 s; `Rv=0.05` | Capacitor-based GFM case |
| 9 | Circular limiter | Notebook-local July model | Zero-voltage fault; recovery at -60 degrees | `exp1_260721`, run `CL` |
| 10 | AVI 0 degrees | Notebook-local July model | Zero-voltage fault; recovery at -60 degrees | `exp1_260721`, run `AVI 0` |
| 11 | AVI 45 degrees | Notebook-local July model | Zero-voltage fault; recovery at -60 degrees | `exp1_260721`, run `AVI 45` |
| 12 | AVI 60 degrees | Notebook-local July model | Zero-voltage fault; recovery at -60 degrees | `exp1_260721`, run `AVI 60` |
| 13 | First-image adaptive-VI replication | `GFMSMIBAVI` with current-device `GFMAVI` | Zero-voltage fault at 0.2 s; recovery at -60 degrees | Reproduces the first MATLAB figure with embedded capacitor and `Rg/Lg` |

Sections 3.1 and 3.2 use `VSGSMIBSystemNoCapacitor`: the converter is a current device, the PCC has no capacitor, and the grid `Rg/Lg` are embedded in the converter. Sections 3.3-3.5 replay the archived, current-limit-disabled `VSGCL` equations used to create those older MATLAB references. All figures are interactive Plotly figures and are not saved.

In [8]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / 'codesimpy').is_dir():
    ROOT = next(parent for parent in Path.cwd().resolve().parents if (parent / 'codesimpy').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from codesimpy.systems import VSGSMIBSystemNoCapacitor

REFERENCE_DIR = Path(r'C:\Users\hange\OneDrive - KTH\ADVOCANO\-AVCI\transavci260410\results\Single_20261005_233637')
REFERENCE_MAT = REFERENCE_DIR / 'result.mat'
OUTPUT_DIR = ROOT / 'results' / 'Single_20261005_233637_codesim'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

WindowsPath('C:/Users/hange/OneDrive - KTH/ADVOCANO/codesim/results/Single_20261005_233637_codesim')

## 3.1 Fixed-time current limiting

The values below were read from `result.mat`. The saved run calibrated `Vs` to `1.1852567951 pu`, applied the voltage ramp from 1.0 to 0.25 pu over 1 ms at 0.2 s, and cleared it with a 1 ms recovery ramp beginning at 0.5 s.

In [9]:
REFERENCE_SETTINGS = {
    'Pref': 1.0,
    'Vs': 1.185256795101828,
    'Kp': 0.05,
    'Lv': 0.3,
    'Lf': 0.07,
    'Rv': 0.1,
    'Imax': 1.5,
    'Kp_cc': 20.0,
    'Ki_cc': 5.0,
    'Vgrid': 1.0,
    'SCR': 2.0,
    'rho': 0.05,
    'faultDepth': 0.75,
    'faultStart_s': 0.2,
    'faultClearTime_s': 0.5,
    'faultRamp_s': 0.001,
}

SYSTEM_OVERRIDES = {
    'SCR': REFERENCE_SETTINGS['SCR'],
    'rho': REFERENCE_SETTINGS['rho'],
    'Pref': REFERENCE_SETTINGS['Pref'],
    'Vref': REFERENCE_SETTINGS['Vs'],
    'clearMode': 'time',
    'faultStart_s': REFERENCE_SETTINGS['faultStart_s'],
    'faultComplete_s': REFERENCE_SETTINGS['faultStart_s'] + REFERENCE_SETTINGS['faultRamp_s'],
    'recoveryStart_s': REFERENCE_SETTINGS['faultClearTime_s'],
    'recoveryComplete_s': REFERENCE_SETTINGS['faultClearTime_s'] + REFERENCE_SETTINGS['faultRamp_s'],
    'faultVoltage': 1.0 - REFERENCE_SETTINGS['faultDepth'],
    'validationTime_s': 1.0,
    'outputStep_s': 1e-4,
    'residualTolerance': 1e-4,
    'Lf': REFERENCE_SETTINGS['Lf'],
    'Rv': REFERENCE_SETTINGS['Rv'],
    'Lv': REFERENCE_SETTINGS['Lv'],
    'Imax': REFERENCE_SETTINGS['Imax'],
    'Kp': REFERENCE_SETTINGS['Kp'],
    'Kp_cc': REFERENCE_SETTINGS['Kp_cc'],
    'Ki_cc': REFERENCE_SETTINGS['Ki_cc'],
}

def create_reference_system():
    parameters = VSGSMIBSystemNoCapacitor.defaultParameters(SYSTEM_OVERRIDES)
    system = VSGSMIBSystemNoCapacitor(parameters)
    return system, parameters

system, parameters = create_reference_system()
{name: system.devices['vsg'].p[name]['num'] for name in ('Lf', 'Rv', 'Lv', 'Rg', 'Lg', 'Imax', 'Kp', 'Kp_cc', 'Ki_cc')}

{'Lf': 0.07,
 'Rv': 0.1,
 'Lv': 0.3,
 'Rg': np.float64(0.024968808471946116),
 'Lg': np.float64(0.4993761694389223),
 'Imax': 1.5,
 'Kp': 0.05,
 'Kp_cc': 20.0,
 'Ki_cc': 5.0}

## Initialize and simulate

SciPy's BDF integrator is used as the closest available counterpart to MATLAB `ode15s`, with the saved tolerances and normalized maximum step.

In [10]:
operating_point = system.initialize(ResidualTolerance=parameters['residualTolerance'])
time_s = np.unique(np.r_[
    np.arange(0.0, parameters['validationTime_s'], parameters['outputStep_s']),
    parameters['validationTime_s'],
    np.asarray(parameters['breakpoints_s'], dtype=float),
])
simulation = system.simulate(
    time_s,
    InitialState=operating_point['state'],
    SolverSettings={
        'method': 'BDF',
        'max_step': 0.02 / parameters['wBase'],
        'rtol': 1e-4,
        'atol': 1e-5,
    },
    Breakpoints_s=parameters['breakpoints_s'],
    saveSnapshot=False,
)
simulation['initialization'] = operating_point['initialization']
print(f"Simulated {time_s.size:,} samples in {simulation['simulationTime_s']:.2f} s")
print(f"Initialization residual: {operating_point['initialization']['residualInfNorm']:.3e}")

Simulated 10,001 samples in 2.33 s
Initialization residual: 3.701e-15


In [11]:
state = simulation['state']
names = system.stateNames
columns = {name: names.index(f'{name}_vsg') for name in (
    'irefd', 'irefq', 'iLd', 'iLq', 'delta'
)}
iref = state[:, [columns['irefd'], columns['irefq']]]
il = state[:, [columns['iLd'], columns['iLq']]]
ireflim = simulation['signals']['irefDQ']
voltage = simulation['signals']['pccVoltage']
active_power = simulation['signals']['activePower'][:, 0]
reactive_power = simulation['signals']['reactivePower'][:, 0]
delta = np.unwrap(state[:, columns['delta']])
delta_rate_per_tau = parameters['Kp'] * (parameters['Pref'] - active_power)
grid_voltage = simulation['signals']['gridVoltage']

extrema = {
    'initialVoltage_pu': float(voltage[0]),
    'minimumVoltage_pu': float(np.min(voltage)),
    'maximumVoltage_pu': float(np.max(voltage)),
    'minimumActivePower_pu': float(np.min(active_power)),
    'maximumActivePower_pu': float(np.max(active_power)),
    'maximumLimitedCurrent_pu': float(np.max(np.linalg.norm(ireflim, axis=1))),
}
extrema

{'initialVoltage_pu': 0.9906061297183706,
 'minimumVoltage_pu': 0.5308652984466318,
 'maximumVoltage_pu': 1.1813981052083056,
 'minimumActivePower_pu': -1.5373946188927663,
 'maximumActivePower_pu': 1.0000000000000038,
 'maximumLimitedCurrent_pu': 1.5000000000000004}

## Optional comparison with the saved MATLAB result

MATLAB v7.3 files require `mat73`/`h5py`. If those packages are unavailable, the CodeSim run and exports still complete; only the overlay and numerical comparison are skipped.

In [12]:
reference = None
mat73_error = None
try:
    import mat73
except ImportError as error:
    local_reader = ROOT / 'tmp' / 'h5py_runtime'
    if local_reader.is_dir():
        sys.path.append(str(local_reader))
        try:
            import mat73
        except ImportError as nested_error:
            mat73_error = nested_error
    else:
        mat73_error = error

if REFERENCE_MAT.is_file() and 'mat73' in globals():
    reference = mat73.loadmat(REFERENCE_MAT)['result']
    print('Loaded MATLAB reference:', REFERENCE_MAT)
else:
    print('Reference comparison skipped.', mat73_error or f'Not found: {REFERENCE_MAT}')

Loaded MATLAB reference: C:\Users\hange\OneDrive - KTH\ADVOCANO\-AVCI\transavci260410\results\Single_20261005_233637\result.mat


In [13]:
comparison = {}
reference_traces = {}
if reference is not None:
    rs = reference['signals']

    def interpolate_reference(values, time_key):
        return np.interp(time_s, np.asarray(rs[time_key]) / parameters['wBase'], np.asarray(values))

    reference_traces['voltage'] = interpolate_reference(np.linalg.norm(rs['vc'], axis=1), 'vc_time')
    reference_traces['activePower'] = interpolate_reference(rs['P'], 'P_time')
    reference_traces['reactivePower'] = -interpolate_reference(rs['Q'], 'Q_time')
    reference_traces['irefMagnitude'] = interpolate_reference(np.linalg.norm(rs['ireflim'], axis=1), 'ireflim_time')
    reference_traces['filterCurrentMagnitude'] = interpolate_reference(np.linalg.norm(rs['il'], axis=1), 'il_time')
    reference_delta = interpolate_reference(np.unwrap(rs['delta']), 'delta_time')
    reference_traces['deltaAligned'] = reference_delta - (reference_delta[0] - delta[0])
    reference_traces['deltaRatePerTau'] = interpolate_reference(rs['ddelta'], 'ddelta_time')

    def error_metrics(actual, expected):
        mask = np.isfinite(actual) & np.isfinite(expected)
        error = np.asarray(actual)[mask] - np.asarray(expected)[mask]
        return {
            'rmse': float(np.sqrt(np.mean(error**2))),
            'mae': float(np.mean(np.abs(error))),
            'maximumAbsoluteError': float(np.max(np.abs(error))),
        }

    comparison = {
        'voltage': error_metrics(voltage, reference_traces['voltage']),
        'activePower': error_metrics(active_power, reference_traces['activePower']),
        'reactivePower': error_metrics(reactive_power, reference_traces['reactivePower']),
        'irefMagnitude': error_metrics(np.linalg.norm(ireflim, axis=1), reference_traces['irefMagnitude']),
        'filterCurrentMagnitude': error_metrics(np.linalg.norm(il, axis=1), reference_traces['filterCurrentMagnitude']),
        'deltaAligned': error_metrics(delta, reference_traces['deltaAligned']),
        'deltaRatePerTau': error_metrics(delta_rate_per_tau, reference_traces['deltaRatePerTau']),
    }
comparison

{'voltage': {'rmse': 0.0010286066083983502,
  'mae': 0.00011587650021808706,
  'maximumAbsoluteError': 0.07318305621535082},
 'activePower': {'rmse': 0.003218676706276226,
  'mae': 0.00031365926197510257,
  'maximumAbsoluteError': 0.2288130414814296},
 'reactivePower': {'rmse': 0.0007156205044070901,
  'mae': 0.00026475601969194507,
  'maximumAbsoluteError': 0.008010641139759411},
 'irefMagnitude': {'rmse': 0.0006961336088474817,
  'mae': 0.00020798131159633707,
  'maximumAbsoluteError': 0.0083549332686772},
 'filterCurrentMagnitude': {'rmse': 0.000697914560786128,
  'mae': 0.00020993572295344317,
  'maximumAbsoluteError': 0.008387492269676722},
 'deltaAligned': {'rmse': 0.00030196598798171924,
  'mae': 0.00018755759289979256,
  'maximumAbsoluteError': 0.0011632165809691841},
 'deltaRatePerTau': {'rmse': 0.00016093383531381124,
  'mae': 1.5682963098755132e-05,
  'maximumAbsoluteError': 0.01144065207407148}}

## Plot

In [14]:
fig = make_subplots(
    rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.035,
    specs=[[{}], [{}], [{}], [{}], [{'secondary_y': True}]],
)

def add_line(row, y, name, *, color=None, dash='solid', secondary_y=False):
    fig.add_trace(
        go.Scatter(
            x=time_s, y=y, mode='lines', name=name,
            line={'color': color, 'dash': dash} if color else {'dash': dash},
        ),
        row=row, col=1, secondary_y=secondary_y,
    )

add_line(1, voltage, 'CodeSim PCC instantaneous', color='black')
add_line(1, grid_voltage, 'Infinite bus', color='#e5a000')
if reference_traces:
    add_line(1, reference_traces['voltage'], 'MATLAB PCC reference', color='#d84a00', dash='dash')

add_line(2, ireflim[:, 0], 'i_ref,d (limited)')
add_line(2, ireflim[:, 1], 'i_ref,q (limited)')
add_line(2, il[:, 0], 'i_L,d', dash='dash')
add_line(2, il[:, 1], 'i_L,q', dash='dash')

add_line(3, np.linalg.norm(ireflim, axis=1), '|i_ref,lim|')
add_line(3, np.linalg.norm(il, axis=1), '|i_L|')
if reference_traces:
    add_line(3, reference_traces['irefMagnitude'], 'MATLAB |i_ref,lim|', dash='dash')

add_line(4, active_power, 'P')
add_line(4, reactive_power, 'Q (injection)')
if reference_traces:
    add_line(4, reference_traces['activePower'], 'MATLAB P', dash='dash')
    add_line(4, reference_traces['reactivePower'], 'MATLAB Q (injection)', dash='dash')

add_line(5, delta - delta[0], 'Delta delta', color='#006dcc')
if reference_traces:
    add_line(5, reference_traces['deltaAligned'] - reference_traces['deltaAligned'][0], 'MATLAB Delta delta', color='#4aa3ff', dash='dash')
add_line(5, delta_rate_per_tau, 'd delta / d tau', color='#e04b00', secondary_y=True)

fig.update_yaxes(title_text='|V| (pu)', row=1, col=1)
fig.update_yaxes(title_text='Current (pu)', row=2, col=1)
fig.update_yaxes(title_text='Current magnitude (pu)', row=3, col=1)
fig.update_yaxes(title_text='P, Q (pu)', row=4, col=1)
fig.update_yaxes(title_text='Delta delta (rad)', row=5, col=1, secondary_y=False)
fig.update_yaxes(title_text='d delta / d tau', row=5, col=1, secondary_y=True)
fig.update_xaxes(title_text='Time (s)', row=5, col=1)
fig.update_layout(
    title='CodeSim replication of Single_20261005_233637',
    height=1300, template='plotly_white', hovermode='x unified',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.01, 'xanchor': 'left', 'x': 0},
)
fig.show()

## 3.2 Single current limiting - 0.25 pu fault, recover at angle

The two supplied MATLAB files contain signal-identical runs. The fault begins at 0.2 s, falls to 0.25 pu, and recovers when the VSG angle reaches 43.544792 degrees. CodeSim detects its own angle crossing; the MATLAB clearing instant is retained for comparison.

In [15]:
from scipy.integrate import solve_ivp
from scipy.io import loadmat
import mat73

def interpolate_columns(source_time, values, target_time):
    values = np.asarray(values)
    if values.ndim == 1:
        return np.interp(target_time, source_time, values)
    return np.column_stack([
        np.interp(target_time, source_time, values[:, column])
        for column in range(values.shape[1])
    ])

def comparison_metrics(actual, expected):
    result = {}
    for field in ('voltage', 'current', 'activePower', 'reactivePower', 'delta'):
        expected_value = interpolate_columns(
            expected['time_s'], expected[field], actual['time_s'],
        )
        actual_value = np.asarray(actual[field])
        if field == 'delta':
            expected_value = expected_value - (expected_value[0] - actual_value[0])
        error = actual_value - expected_value
        result[field] = {
            'rmse': float(np.sqrt(np.mean(error**2))),
            'maximumAbsoluteError': float(np.max(np.abs(error))),
        }
    result['clearingTimeError_s'] = float(
        actual['clearingTime_s'] - expected['clearingTime_s']
    )
    return result

def draw_replication(title, actual, expected, fault_start_s, target_angle_rad=None):
    figure = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.045)
    colors = {'voltage': '#006dcc', 'grid': '#e07a00', 'd': '#006dcc', 'q': '#d84a00'}

    def line(row, trace, field, name, color, dash='solid', column=None):
        value = np.asarray(trace[field])
        if column is not None:
            value = value[:, column]
        figure.add_trace(go.Scatter(
            x=trace['time_s'], y=value, mode='lines', name=name,
            line={'color': color, 'dash': dash},
        ), row=row, col=1)

    line(1, actual, 'voltage', 'CodeSim PCC', colors['voltage'])
    line(1, actual, 'gridVoltage', 'CodeSim infinite bus', colors['grid'])
    line(1, expected, 'voltage', 'MATLAB PCC', colors['voltage'], 'dash')
    line(1, expected, 'gridVoltage', 'MATLAB infinite bus', colors['grid'], 'dash')
    line(2, actual, 'current', 'CodeSim i_d', colors['d'], column=0)
    line(2, actual, 'current', 'CodeSim i_q', colors['q'], column=1)
    line(2, expected, 'current', 'MATLAB i_d', colors['d'], 'dash', 0)
    line(2, expected, 'current', 'MATLAB i_q', colors['q'], 'dash', 1)
    line(3, actual, 'activePower', 'CodeSim P', colors['d'])
    line(3, actual, 'reactivePower', 'CodeSim Q', colors['q'])
    line(3, expected, 'activePower', 'MATLAB P', colors['d'], 'dash')
    line(3, expected, 'reactivePower', 'MATLAB Q', colors['q'], 'dash')
    line(4, actual, 'delta', 'CodeSim delta', colors['d'])
    line(4, expected, 'delta', 'MATLAB delta', colors['q'], 'dash')

    for row in range(1, 5):
        figure.add_vline(x=fault_start_s, line_dash='dot', line_color='#555', row=row, col=1)
        figure.add_vline(x=actual['clearingTime_s'], line_dash='dashdot', line_color='#006dcc', row=row, col=1)
        figure.add_vline(x=expected['clearingTime_s'], line_dash='dashdot', line_color='#d84a00', row=row, col=1)
    if target_angle_rad is not None:
        figure.add_hline(y=target_angle_rad, line_dash='dot', line_color='#555', row=4, col=1)

    figure.update_yaxes(title_text='|V| (pu)', row=1, col=1)
    figure.update_yaxes(title_text='i_d, i_q (pu)', row=2, col=1)
    figure.update_yaxes(title_text='P, Q (pu)', row=3, col=1)
    figure.update_yaxes(title_text='delta (rad)', row=4, col=1)
    figure.update_xaxes(title_text='Time (s)', row=4, col=1)
    figure.update_layout(
        title=title, height=1050, template='plotly_white', hovermode='x unified',
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.01},
    )
    figure.show()

def load_current_limited_reference(path):
    result = mat73.loadmat(path)['result']
    settings = result['settings']
    signals = result['signals']
    w_base = 2 * np.pi * 50
    time_s = np.asarray(signals['vc_time']) / w_base
    return result, {
        'time_s': time_s,
        'voltage': np.linalg.norm(signals['vc'], axis=1),
        'gridVoltage': interpolate_columns(
            np.asarray(signals['vg_time']) / w_base, signals['vg'], time_s,
        ),
        'current': interpolate_columns(
            np.asarray(signals['ireflim_time']) / w_base, signals['ireflim'], time_s,
        ),
        'activePower': interpolate_columns(
            np.asarray(signals['P_time']) / w_base, signals['P'], time_s,
        ),
        'reactivePower': -interpolate_columns(
            np.asarray(signals['Q_time']) / w_base, signals['Q'], time_s,
        ),
        'delta': interpolate_columns(
            np.asarray(signals['delta_time']) / w_base, np.unwrap(signals['delta']), time_s,
        ),
        'clearingTime_s': float(result['clearingTime_s']),
        'settings': settings,
    }

CURRENT_LIMITING_PATHS = [
    Path(r'C:\Users\hange\OneDrive - KTH\ADVOCANO\-AVCI\transavci260410\results\Single_20261005_232639\result.mat'),
    Path(r'C:\Users\hange\OneDrive - KTH\ADVOCANO\-AVCI\transavci260410\results\Single_20261003_230407\result.mat'),
]
current_results = [load_current_limited_reference(path) for path in CURRENT_LIMITING_PATHS]
current_reference_result, current_reference = current_results[0]
duplicate_maximum_error = max(
    float(np.max(np.abs(
        np.asarray(current_results[0][0]['signals'][field])
        - np.asarray(current_results[1][0]['signals'][field])
    )))
    for field in current_results[0][0]['signals']
)

settings_32 = current_reference['settings']
parameters_32 = VSGSMIBSystemNoCapacitor.defaultParameters({
    'SCR': float(settings_32['SCR']), 'rho': float(settings_32['rho']),
    'Pref': float(settings_32['Pref']), 'Vref': float(settings_32['Vs']),
    'Lf': float(settings_32['Lf']), 'Rv': float(settings_32['Rv']),
    'Lv': float(settings_32['Lv']), 'Imax': float(settings_32['Imax']),
    'Kp': float(settings_32['Kp']), 'Kp_cc': float(settings_32['Kp_cc']),
    'Ki_cc': float(settings_32['Ki_cc']), 'clearMode': 'angle',
    'clearAngle_rad': np.deg2rad(float(settings_32['clearAngle_deg'])),
    'minimumFaultDuration_s': float(settings_32['minFaultTime_s']),
    'faultStart_s': float(settings_32['faultStart_s']),
    'faultComplete_s': float(settings_32['faultStart_s'] + settings_32['faultRamp_s']),
    'faultVoltage': 1 - float(settings_32['faultDepth']),
    'validationTime_s': float(current_reference_result['stopTime_s']),
    'outputStep_s': 1e-4,
})
system_32 = VSGSMIBSystemNoCapacitor(parameters_32)
operating_point_32 = system_32.initialize(
    ResidualTolerance=parameters_32['residualTolerance'],
)
time_32 = np.unique(np.r_[
    np.arange(0, parameters_32['validationTime_s'], parameters_32['outputStep_s']),
    parameters_32['validationTime_s'], parameters_32['breakpoints_s'],
])
simulation_32 = system_32.simulate(
    time_32, InitialState=operating_point_32['state'],
    SolverSettings={'method': 'BDF', 'max_step': 0.02 / parameters_32['wBase'], 'rtol': 1e-4, 'atol': 1e-5},
    Breakpoints_s=parameters_32['breakpoints_s'], saveSnapshot=False,
)
vsg_32 = system_32.devices['vsg']
delta_index_32 = system_32.stateNames.index(f'delta{vsg_32.index}')
current_case = {
    'time_s': simulation_32['time_s'],
    'voltage': simulation_32['signals']['pccVoltage'],
    'gridVoltage': simulation_32['signals']['gridVoltage'],
    'current': simulation_32['signals']['irefDQ'],
    'activePower': simulation_32['signals']['activePower'][:, 0],
    'reactivePower': simulation_32['signals']['reactivePower'][:, 0],
    'delta': np.unwrap(simulation_32['state'][:, delta_index_32]),
    'clearingTime_s': float(simulation_32['clearingTime_s']),
}
metrics_32 = comparison_metrics(current_case, current_reference)
print('Maximum difference between the two supplied MATLAB runs:', duplicate_maximum_error)
print('CodeSim / MATLAB clearing times:', current_case['clearingTime_s'], current_reference['clearingTime_s'])
print(metrics_32)

draw_replication(
    '3.2 Single current limiting: V_f = 0.25 pu, angle recovery',
    current_case, current_reference, parameters_32['faultStart_s'],
    parameters_32['clearAngle_rad'],
)

Maximum difference between the two supplied MATLAB runs: 0.0
CodeSim / MATLAB clearing times: 0.6468906871833552 0.6463472092652839
{'voltage': {'rmse': 0.011192462655217403, 'maximumAbsoluteError': 0.22820178016551218}, 'current': {'rmse': 0.026031493274796372, 'maximumAbsoluteError': 0.28180621054946475}, 'activePower': {'rmse': 0.026553471636338366, 'maximumAbsoluteError': 0.4725715726047412}, 'reactivePower': {'rmse': 0.026826212751497193, 'maximumAbsoluteError': 0.2831378284500973}, 'delta': {'rmse': 0.005547662934248417, 'maximumAbsoluteError': 0.0132580408992089}, 'clearingTimeError_s': 0.0005434779180713001}


## Archived VSGCL replay used by sections 3.3-3.5

These September references predate the no-capacitor current-device model. Their archived source uses an LC-filter `VSGCL`, an explicit grid branch, and `Zv = 0` (current limitation disabled). The helper below replays those exact normalized dq equations from the saved initial state. It does not modify any registered CodeSim system.

In [ ]:
HISTORICAL_ROOT = Path(r'C:\Users\hange\OneDrive - KTH\ADVOCANO\-AVCI\transavci260410\results\VSG_SMIB_20260908_115547')

def historical_reference(path):
    result = loadmat(path, simplify_cells=True)['result']
    signals = result['signals']
    return result, {
        'time_s': np.asarray(result['time_s']),
        'voltage': np.asarray(signals['busVoltageMagnitude'])[:, 0],
        'gridVoltage': np.asarray(signals['gridVoltageReference']),
        'current': np.asarray(signals['irefDQ']),
        'activePower': np.asarray(signals['activePower'])[:, 0],
        'reactivePower': np.asarray(signals['reactivePower'])[:, 0],
        'delta': np.asarray(signals['deltaRaw']),
        'clearingTime_s': float(result['metrics']['clearTime_s']),
    }

def replay_historical_vsgcl(result):
    p = result['parameters']
    reference_state = np.asarray(result['state'][0], dtype=float)
    theta_vsg, theta_grid = reference_state[7], reference_state[8]
    rotation = np.array([
        [np.cos(theta_vsg), -np.sin(theta_vsg)],
        [np.sin(theta_vsg), np.cos(theta_vsg)],
    ])
    initial_state = np.r_[
        rotation.T @ reference_state[:2],
        reference_state[2:7],
    ]
    grid_angle = theta_grid - theta_vsg
    grid_direction = np.array([np.cos(grid_angle), np.sin(grid_angle)])
    J = np.array([[0.0, 1.0], [-1.0, 0.0]])

    def derivative(time_s, state, clearing_time_s):
        fall = np.clip((time_s - p['faultStart_s']) / p['rampTime_s'], 0, 1)
        rise = (
            0.0 if not np.isfinite(clearing_time_s)
            else np.clip((time_s - clearing_time_s) / p['rampTime_s'], 0, 1)
        )
        grid_magnitude = 1 - p['faultDepth'] * fall + p['faultDepth'] * rise
        grid_voltage = grid_magnitude * grid_direction
        grid_current = state[0:2]
        pcc_voltage = state[2:4]
        filter_current = state[4:6]
        delta = state[6]
        internal_voltage = p['Vref'] * np.array([np.cos(delta), np.sin(delta)])
        d_grid_current = (
            (pcc_voltage - grid_voltage - p['Rg'] * grid_current) / p['Lg']
            + J @ grid_current
        )
        d_pcc_voltage = (filter_current - grid_current) / p['C'] - J @ pcc_voltage
        d_filter_current = (
            internal_voltage - pcc_voltage - p['Rf'] * filter_current
            + p['Lf'] * J @ filter_current
        ) / p['Lf']
        d_delta = p['Kp'] * (p['Pref'] - pcc_voltage @ grid_current)
        return p['wBase'] * np.r_[
            d_grid_current, d_pcc_voltage, d_filter_current, d_delta,
        ]

    solver = {'method': 'DOP853', 'rtol': 1e-9, 'atol': 1e-11, 'max_step': 0.01 / p['wBase'], 'dense_output': True}
    segments = []
    if np.isfinite(p['clearAngle_deg']):
        event_start = p['faultStart_s'] + p['rampTime_s']
        before_event = solve_ivp(
            lambda t, x: derivative(t, x, np.inf),
            (0, event_start), initial_state, **solver,
        )
        target = np.deg2rad(p['clearAngle_deg'])
        def angle_event(_time_s, state):
            return np.sin((state[6] - target) / 2)
        angle_event.terminal = True
        angle_event.direction = 0
        to_event = solve_ivp(
            lambda t, x: derivative(t, x, np.inf),
            (event_start, p['latestClearTime_s']), before_event.y[:, -1],
            events=angle_event, **solver,
        )
        if not to_event.t_events[0].size:
            raise RuntimeError('Historical angle target was not reached')
        clearing_time_s = float(to_event.t_events[0][0])
        clearing_state = to_event.y_events[0][0]
        segments.extend([before_event, to_event])
    else:
        clearing_time_s = float(p['clearTime_s'])
        before_event = solve_ivp(
            lambda t, x: derivative(t, x, np.inf),
            (0, clearing_time_s), initial_state, **solver,
        )
        clearing_state = before_event.y[:, -1]
        segments.append(before_event)

    final_time_s = float(np.asarray(result['time_s'])[-1])
    after_event = solve_ivp(
        lambda t, x: derivative(t, x, clearing_time_s),
        (clearing_time_s, final_time_s), clearing_state, **solver,
    )
    segments.append(after_event)
    time_s = np.asarray(result['time_s'])
    state = np.empty((time_s.size, initial_state.size))
    start = 0.0
    for index, segment in enumerate(segments):
        stop = segment.t[-1]
        mask = (time_s >= start) & (time_s <= stop if index == len(segments) - 1 else time_s < stop)
        state[mask] = segment.sol(time_s[mask]).T
        start = stop

    fall = np.clip((time_s - p['faultStart_s']) / p['rampTime_s'], 0, 1)
    rise = np.clip((time_s - clearing_time_s) / p['rampTime_s'], 0, 1)
    pcc_voltage, grid_current = state[:, 2:4], state[:, 0:2]
    return {
        'time_s': time_s,
        'voltage': np.linalg.norm(pcc_voltage, axis=1),
        'gridVoltage': 1 - p['faultDepth'] * fall + p['faultDepth'] * rise,
        'current': state[:, 4:6],
        'activePower': np.sum(pcc_voltage * grid_current, axis=1),
        'reactivePower': pcc_voltage[:, 0] * grid_current[:, 1] - pcc_voltage[:, 1] * grid_current[:, 0],
        'delta': state[:, 6],
        'clearingTime_s': clearing_time_s,
    }

def run_historical_case(relative_path, title):
    result, expected = historical_reference(HISTORICAL_ROOT / relative_path)
    actual = replay_historical_vsgcl(result)
    metrics = comparison_metrics(actual, expected)
    target = result['parameters']['clearAngle_deg']
    draw_replication(
        title, actual, expected, result['parameters']['faultStart_s'],
        None if not np.isfinite(target) else np.deg2rad(target),
    )
    return result, actual, expected, metrics

## 3.3 Overvoltage - recover at -90 degrees

In [17]:
result_33, case_33, reference_33, metrics_33 = run_historical_case(
    Path('G2_angle') / 'G2_angle_m90.mat',
    '3.3 Overvoltage: recover at -90 degrees',
)
metrics_33

{'voltage': {'rmse': 0.001258518140370146,
  'maximumAbsoluteError': 0.013456464555013148},
 'current': {'rmse': 0.0035625435689094266,
  'maximumAbsoluteError': 0.03073601432155848},
 'activePower': {'rmse': 0.005621526323475482,
  'maximumAbsoluteError': 0.0534469452994516},
 'reactivePower': {'rmse': 0.0044733576716030725,
  'maximumAbsoluteError': 0.03568726455878224},
 'delta': {'rmse': 0.00102005904695421,
  'maximumAbsoluteError': 0.005313854252810835},
 'clearingTimeError_s': -3.199854413654668e-05}

## 3.4 Undervoltage - recover at +90 degrees

In [18]:
result_34, case_34, reference_34, metrics_34 = run_historical_case(
    Path('G2_angle') / 'G2_angle_90.mat',
    '3.4 Undervoltage: recover at +90 degrees',
)
metrics_34

{'voltage': {'rmse': 0.0007309825971994225,
  'maximumAbsoluteError': 0.009126207194308011},
 'current': {'rmse': 0.0037352302498403073,
  'maximumAbsoluteError': 0.031768501136908656},
 'activePower': {'rmse': 0.0027467784415086864,
  'maximumAbsoluteError': 0.028384254241087592},
 'reactivePower': {'rmse': 0.004307682512892073,
  'maximumAbsoluteError': 0.04376319304110954},
 'delta': {'rmse': 0.001010467302930581,
  'maximumAbsoluteError': 0.0026525683118316934},
 'clearingTimeError_s': -3.199857511765325e-05}

## 3.5 Undervoltage followed by overvoltage - 80% depth, fixed recovery

In [19]:
result_35, case_35, reference_35, metrics_35 = run_historical_case(
    Path('G4_depth') / 'G4_depth_0p8.mat',
    '3.5 Undervoltage followed by overvoltage: 80% depth',
)
replication_summary = {
    '3.2 current limiting': metrics_32,
    '3.3 overvoltage': metrics_33,
    '3.4 undervoltage': metrics_34,
    '3.5 undervoltage then overvoltage': metrics_35,
}
replication_summary

{'3.2 current limiting': {'voltage': {'rmse': 0.011192462655217403,
   'maximumAbsoluteError': 0.22820178016551218},
  'current': {'rmse': 0.026031493274796372,
   'maximumAbsoluteError': 0.28180621054946475},
  'activePower': {'rmse': 0.026553471636338366,
   'maximumAbsoluteError': 0.4725715726047412},
  'reactivePower': {'rmse': 0.026826212751497193,
   'maximumAbsoluteError': 0.2831378284500973},
  'delta': {'rmse': 0.005547662934248417,
   'maximumAbsoluteError': 0.0132580408992089},
  'clearingTimeError_s': 0.0005434779180713001},
 '3.3 overvoltage': {'voltage': {'rmse': 0.001258518140370146,
   'maximumAbsoluteError': 0.013456464555013148},
  'current': {'rmse': 0.0035625435689094266,
   'maximumAbsoluteError': 0.03073601432155848},
  'activePower': {'rmse': 0.005621526323475482,
   'maximumAbsoluteError': 0.0534469452994516},
  'reactivePower': {'rmse': 0.0044733576716030725,
   'maximumAbsoluteError': 0.03568726455878224},
  'delta': {'rmse': 0.00102005904695421,
   'maximumAb

# Additional GFL and GFM cases

These cases use the refactored CodeSim systems directly. All figures are interactive Plotly figures and are not saved to disk.

In [20]:
from codesimpy.systems import (
    GFLLVRTSystem, GFLLVRTSystemCapacitor, VSGSMIBSystem,
    VSGSMIBSystemNoCapacitor,
)

def simulate_system(system_class, overrides=None, device_controls=None, solver_settings=None):
    parameters = system_class.defaultParameters(overrides or {})
    system = system_class(parameters)
    for device_name, values in (device_controls or {}).items():
        for parameter_name, value in values.items():
            system.devices[device_name].p[parameter_name]['num'] = value
    operating_point = system.initialize(ResidualTolerance=parameters['residualTolerance'])
    time_s = np.unique(np.r_[
        np.arange(0, parameters['validationTime_s'], parameters['outputStep_s']),
        parameters['validationTime_s'], parameters['breakpoints_s'],
    ])
    result = system.simulate(
        time_s, InitialState=operating_point['state'],
        SolverSettings=solver_settings or system.solverSettings,
        Breakpoints_s=parameters['breakpoints_s'], saveSnapshot=False,
    )
    return system, parameters, result

def count_rising_edges(active, time_s, start_s=0.0):
    active = np.asarray(active, dtype=bool)
    time_s = np.asarray(time_s)
    return int(np.sum((~active[:-1]) & active[1:] & (time_s[1:] >= start_s)))


## 1. GFLLVRTSystem - temporary overvoltage

The system is simulated with its default parameters. The post-fault PCC overshoot is shown together with the grid voltage, converter current, power, and FRT state.

In [21]:
gfl_default_system, gfl_default_parameters, gfl_default_result = simulate_system(GFLLVRTSystem)
gfl_default_signals = gfl_default_result['signals']
gfl_default_time = gfl_default_result['time_s']

figure_gfl_overvoltage = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.045,
    row_heights=[0.34, 0.24, 0.25, 0.17],
)
figure_gfl_overvoltage.add_trace(go.Scatter(
    x=gfl_default_time, y=gfl_default_signals['pccVoltage'], name='PCC voltage',
    line={'color': '#006dcc'},
), row=1, col=1)
figure_gfl_overvoltage.add_trace(go.Scatter(
    x=gfl_default_time, y=gfl_default_signals['gridVoltage'], name='Infinite-bus voltage',
    line={'color': '#e07a00'},
), row=1, col=1)
figure_gfl_overvoltage.add_hline(y=1.0, line_dash='dot', line_color='#777', row=1, col=1)
figure_gfl_overvoltage.add_trace(go.Scatter(
    x=gfl_default_time, y=gfl_default_signals['gflCurrentMagnitude'], name='GFL current',
    line={'color': '#00876c'},
), row=2, col=1)
figure_gfl_overvoltage.add_trace(go.Scatter(
    x=gfl_default_time, y=gfl_default_signals['activePower'], name='P',
    line={'color': '#006dcc'},
), row=3, col=1)
figure_gfl_overvoltage.add_trace(go.Scatter(
    x=gfl_default_time, y=gfl_default_signals['reactivePower'], name='Q',
    line={'color': '#d84a00'},
), row=3, col=1)
figure_gfl_overvoltage.add_trace(go.Scatter(
    x=gfl_default_time, y=gfl_default_signals['lvrtMode'], name='FRT active',
    line={'color': '#8e44ad', 'shape': 'hv'},
), row=4, col=1)
for row in range(1, 5):
    figure_gfl_overvoltage.add_vrect(
        x0=gfl_default_parameters['faultStart_s'], x1=gfl_default_parameters['recoveryStart_s'],
        fillcolor='#777', opacity=0.08, line_width=0, row=row, col=1,
    )
figure_gfl_overvoltage.update_yaxes(title_text='|V| (pu)', row=1, col=1)
figure_gfl_overvoltage.update_yaxes(title_text='|i| (pu)', row=2, col=1)
figure_gfl_overvoltage.update_yaxes(title_text='P, Q (pu)', row=3, col=1)
figure_gfl_overvoltage.update_yaxes(title_text='FRT', tickvals=[0, 1], row=4, col=1)
figure_gfl_overvoltage.update_xaxes(title_text='Time (s)', row=4, col=1)
figure_gfl_overvoltage.update_layout(
    title='GFLLVRTSystem: temporary overvoltage (default parameters)',
    height=900, template='plotly_white', hovermode='x unified',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.01},
)
post_recovery_gfl = gfl_default_time >= gfl_default_parameters['recoveryStart_s']
print('Peak post-recovery PCC voltage:', float(np.max(gfl_default_signals['pccVoltage'][post_recovery_gfl])), 'pu')
figure_gfl_overvoltage.show(config={'displaylogo': False})


Peak post-recovery PCC voltage: 1.0912982916446166 pu


## 2. GFLLVRTSystemCapacitor - recurrent FRT

Low capacitor damping exposes the LC-mode oscillation and repeated FRT threshold crossings. Only short windows around fault entry and recovery are plotted.

In [22]:
gfl_cap_overrides = {
    'SCR': 1.5, 'rho': 0.1, 'Pref': 1.0, 'Qref': 0.0,
    'C': 0.068, 'K1': 4.0, 'validationTime_s': 0.8,
    'faultStart_s': 0.2, 'faultComplete_s': 0.201,
    'recoveryStart_s': 0.35, 'recoveryComplete_s': 0.351,
    'faultVoltage': 0.35, 'outputStep_s': 2e-4,
}
gfl_cap_controls = {
    'gfl': {'K1': 4.0, 'KpPll': 0.2, 'KiPll': 0.2},
    'capacitor': {'Gd': 0.25, 'C': 0.068},
}
gfl_cap_system, gfl_cap_parameters, gfl_cap_result = simulate_system(
    GFLLVRTSystemCapacitor, gfl_cap_overrides, gfl_cap_controls,
    {'method': 'DOP853', 'max_step': 2e-5, 'rtol': 2e-5, 'atol': 1e-8},
)
gfl_cap_signals = gfl_cap_result['signals']
gfl_cap_time = gfl_cap_result['time_s']
gfl_cap_windows = [
    ('Fault entry', gfl_cap_parameters['faultStart_s'] - 0.005, gfl_cap_parameters['faultStart_s'] + 0.045),
    ('Fault recovery', gfl_cap_parameters['recoveryStart_s'] - 0.005, gfl_cap_parameters['recoveryStart_s'] + 0.045),
]
figure_gfl_cap = make_subplots(
    rows=3, cols=2, shared_xaxes='columns', vertical_spacing=0.08, horizontal_spacing=0.09,
    column_titles=[window[0] for window in gfl_cap_windows],
)
for column, (_, window_start, window_end) in enumerate(gfl_cap_windows, start=1):
    mask = (gfl_cap_time >= window_start) & (gfl_cap_time <= window_end)
    show_legend = column == 1
    figure_gfl_cap.add_trace(go.Scatter(
        x=gfl_cap_time[mask], y=gfl_cap_signals['pccVoltage'][mask], name='PCC voltage',
        legendgroup='pcc', showlegend=show_legend, line={'color': '#006dcc'},
    ), row=1, col=column)
    figure_gfl_cap.add_trace(go.Scatter(
        x=gfl_cap_time[mask], y=gfl_cap_signals['gridVoltage'][mask], name='Infinite-bus voltage',
        legendgroup='grid', showlegend=show_legend, line={'color': '#e07a00'},
    ), row=1, col=column)
    figure_gfl_cap.add_hline(y=0.9, line_dash='dot', line_color='#555', row=1, col=column)
    figure_gfl_cap.add_trace(go.Scatter(
        x=gfl_cap_time[mask], y=gfl_cap_signals['gflCurrentMagnitude'][mask], name='Converter current',
        legendgroup='current', showlegend=show_legend, line={'color': '#00876c'},
    ), row=2, col=column)
    figure_gfl_cap.add_trace(go.Scatter(
        x=gfl_cap_time[mask], y=gfl_cap_signals['lvrtMode'][mask], name='FRT active',
        legendgroup='frt', showlegend=show_legend, line={'color': '#8e44ad', 'shape': 'hv'},
    ), row=3, col=column)
    figure_gfl_cap.update_xaxes(range=[window_start, window_end], title_text='Time (s)', row=3, col=column)
figure_gfl_cap.update_yaxes(title_text='|V| (pu)', row=1, col=1)
figure_gfl_cap.update_yaxes(title_text='|i| (pu)', row=2, col=1)
figure_gfl_cap.update_yaxes(title_text='FRT', tickvals=[0, 1], range=[-0.08, 1.08], row=3, col=1)
figure_gfl_cap.update_layout(
    title='GFLLVRTSystemCapacitor: recurrent FRT, focused event windows',
    height=760, template='plotly_white', hovermode='x unified',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.08},
)
post_recovery_cap = gfl_cap_time >= gfl_cap_parameters['recoveryStart_s']
print('Post-recovery FRT entries:', count_rising_edges(
    gfl_cap_signals['lvrtMode'] > 0.5, gfl_cap_time, gfl_cap_parameters['recoveryStart_s'],
))
figure_gfl_cap.show(config={'displaylogo': False})


Post-recovery FRT entries: 97


## 3. GFMSMIB - recurrent current limiting

The GFM case uses the capacitor-based `VSGSMIBSystem` with the `VSGCC` device: fault voltage 0.5 pu at 0.2 s, fixed recovery at 0.3 s, and virtual resistance `Rv = 0.05` pu.

In [23]:
gfm_recurrent_overrides = {
    'faultVoltage': 0.5, 'faultStart_s': 0.2, 'faultComplete_s': 0.201,
    'recoveryStart_s': 0.3, 'recoveryComplete_s': 0.301,
    'Rv': 0.05, 'clearMode': 'time', 'validationTime_s': 0.6,
    'outputStep_s': 1e-4,
}
gfm_system, gfm_parameters, gfm_result = simulate_system(
    VSGSMIBSystem, gfm_recurrent_overrides,
    solver_settings={'method': 'BDF', 'max_step': 0.02 / (2 * np.pi * 50), 'rtol': 1e-4, 'atol': 1e-5},
)
gfm_signals = gfm_result['signals']
gfm_time = gfm_result['time_s']
gfm_vsg = gfm_system.devices['vsg']
gfm_reference_indices = [
    gfm_system.stateNames.index(f'irefd{gfm_vsg.index}'),
    gfm_system.stateNames.index(f'irefq{gfm_vsg.index}'),
]
gfm_unlimited_current = np.linalg.norm(gfm_result['state'][:, gfm_reference_indices], axis=1)
gfm_current_limit = float(gfm_vsg.p['Imax']['num'])
gfm_limiter_active = gfm_unlimited_current >= gfm_current_limit

figure_gfm_recurrent = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.045,
    row_heights=[0.34, 0.24, 0.24, 0.18],
)
figure_gfm_recurrent.add_trace(go.Scatter(
    x=gfm_time, y=gfm_signals['busVoltageMagnitude'][:, 0], name='PCC voltage', line={'color': '#006dcc'},
), row=1, col=1)
figure_gfm_recurrent.add_trace(go.Scatter(
    x=gfm_time, y=gfm_signals['busVoltageMagnitude'][:, 1], name='Infinite-bus voltage', line={'color': '#e07a00'},
), row=1, col=1)
figure_gfm_recurrent.add_trace(go.Scatter(
    x=gfm_time, y=gfm_unlimited_current, name='Unlimited current request', line={'color': '#8e44ad'},
), row=2, col=1)
figure_gfm_recurrent.add_hline(
    y=gfm_current_limit, line_dash='dash', line_color='#222', row=2, col=1,
)
figure_gfm_recurrent.add_trace(go.Scatter(
    x=gfm_time, y=gfm_signals['filterCurrentMagnitude'], name='Delivered filter current',
    line={'color': '#00876c'},
), row=3, col=1)
figure_gfm_recurrent.add_hline(
    y=gfm_current_limit, line_dash='dash', line_color='#222', row=3, col=1,
)
figure_gfm_recurrent.add_trace(go.Scatter(
    x=gfm_time, y=gfm_limiter_active.astype(float), name='Limiter active',
    line={'color': '#c0392b', 'shape': 'hv'}, fill='tozeroy',
), row=4, col=1)
for row in range(1, 5):
    figure_gfm_recurrent.add_vrect(
        x0=gfm_parameters['faultStart_s'], x1=gfm_parameters['recoveryStart_s'],
        fillcolor='#777', opacity=0.08, line_width=0, row=row, col=1,
    )
figure_gfm_recurrent.update_yaxes(title_text='|V| (pu)', row=1, col=1)
figure_gfm_recurrent.update_yaxes(title_text='|i_unlim| (pu)', row=2, col=1)
figure_gfm_recurrent.update_yaxes(title_text='|i| (pu)', row=3, col=1)
figure_gfm_recurrent.update_yaxes(title_text='Limit', tickvals=[0, 1], range=[-0.08, 1.08], row=4, col=1)
figure_gfm_recurrent.update_xaxes(title_text='Time (s)', row=4, col=1)
figure_gfm_recurrent.update_layout(
    title='GFM-SMIB with capacitor: recurrent current limiting (Vf=0.5 pu, recovery=0.3 s, Rv=0.05 pu)',
    height=950, template='plotly_white', hovermode='x unified',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.01},
)
print('Limiter entries:', count_rising_edges(gfm_limiter_active, gfm_time))
print('Post-recovery limiter entries:', count_rising_edges(
    gfm_limiter_active, gfm_time, gfm_parameters['recoveryStart_s'],
))
figure_gfm_recurrent.show(config={'displaylogo': False})


Limiter entries: 23
Post-recovery limiter entries: 7


# July 2026 GFM current-limiter replication

This compact, notebook-local replay follows `testCurrentAngle260707.slx` and `computeVoltage260721.m`; it does not change the default CodeSim VSG-SMIB system. The original constants recovered from `exp1_260721.mldatx` are `C=0.1`, `Lf=Lg=0.1`, `Rf=0.1`, `Rg=0.01`, `Vref=1`, `Pref=0`, and `Kp=0.2`. AVI uses `Ith=0.9`, `k=2`, while CL uses `Imax=1.2`. The fault starts at 0.2 s and clears at delta = -60 degrees. Figures are interactive Plotly objects and are not written to disk.


In [24]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

J_DQ = np.array([[0.0, 1.0], [-1.0, 0.0]])
JULY_PARAMETERS = {
    'C': 0.1, 'Lf': 0.1, 'Rf': 0.1, 'Lg': 0.1, 'Rg': 0.01,
    'Vref': 1.0, 'Pref': 0.0, 'Kp': 0.2,
    'Ith': 0.9, 'kVI': 2.0, 'Imax': 1.2,
    'fault_tau': 20 * np.pi, 'clear_angle': -np.pi / 3,
}
INITIAL_STATE = np.array([
    0.995018548430395, -0.000495639105180,
    0.0000247515166, 0.0496898204,
   -0.0000248123939, -0.0498120345,
    0.004470882676369,
])
ARCHIVED_CLEAR_TAU = {
    'AVI 0': 408.819854, 'AVI 45': 427.684951,
    'AVI 60': 429.086651, 'CL': 429.837467,
}
ARCHIVED_PEAK = {'AVI 0': 1.317397, 'AVI 45': 1.334964, 'AVI 60': 1.409280, 'CL': 1.339704}

def july_rhs(_tau, state, grid_voltage, limiter, angle_rad=0.0):
    p = JULY_PARAMETERS
    vc, iref, ig, delta = state[:2], state[2:4], state[4:6], state[6]
    internal_voltage = p['Vref'] * np.array([np.cos(delta), -np.sin(delta)])
    error = internal_voltage - vc
    if limiter == 'AVI':
        excess = max(0.0, np.linalg.norm(iref) - p['Ith'])
        direction = p['kVI'] * (np.cos(angle_rad) * np.eye(2) + np.sin(angle_rad) * J_DQ)
        virtual_impedance = direction * excess
    else:
        filter_impedance = p['Rf'] * np.eye(2) + p['Lf'] * J_DQ
        unlimited = np.linalg.solve(filter_impedance, error)
        excess = max(0.0, np.linalg.norm(unlimited) - p['Imax'])
        virtual_impedance = filter_impedance * excess / p['Imax']
    diref = (error - p['Rf'] * iref - p['Lf'] * J_DQ @ iref - virtual_impedance @ iref) / p['Lf']
    dvc = (iref - ig) / p['C'] + J_DQ @ vc
    vg = np.array([grid_voltage, 0.0])
    dig = (vc - vg - p['Rg'] * ig - p['Lg'] * J_DQ @ ig) / p['Lg']
    ddelta = p['Kp'] * (p['Pref'] - vc @ ig)
    return np.r_[dvc, diref, dig, ddelta]

def simulate_july_case(label, limiter, angle_deg=0.0):
    p = JULY_PARAMETERS
    angle_rad = np.deg2rad(angle_deg)
    common = dict(rtol=1e-7, atol=1e-9, max_step=0.05, dense_output=True)
    pre = solve_ivp(lambda t, x: july_rhs(t, x, 1.0, limiter, angle_rad),
                    (0.0, p['fault_tau']), INITIAL_STATE, **common)
    def clearing_event(_t, x):
        return x[6] - p['clear_angle']
    clearing_event.direction = -1
    clearing_event.terminal = True
    fault = solve_ivp(lambda t, x: july_rhs(t, x, 0.0, limiter, angle_rad),
                      (p['fault_tau'], 300 * np.pi), pre.y[:, -1],
                      events=clearing_event, **common)
    clear_tau = float(fault.t_events[0][0])
    post = solve_ivp(lambda t, x: july_rhs(t, x, 1.0, limiter, angle_rad),
                     (clear_tau, clear_tau + 25.0), fault.y_events[0][0], **common)
    relative_tau = np.arange(-5.0, 20.0001, 0.02)
    state = np.column_stack([
        fault.sol(clear_tau + dt) if dt < 0 else post.sol(clear_tau + dt)
        for dt in relative_tau
    ])
    archive_tau = ARCHIVED_CLEAR_TAU[label]
    return {
        'label': label, 'relative_tau': relative_tau,
        'time_ms': (archive_tau + relative_tau) * 10 / np.pi,
        'relative_ms': relative_tau * 10 / np.pi,
        'state': state, 'clear_tau': clear_tau,
        'grid_voltage': (relative_tau >= 0).astype(float),
    }

july_cases = {
    'CL': simulate_july_case('CL', 'CL'),
    'AVI 0': simulate_july_case('AVI 0', 'AVI', 0),
    'AVI 45': simulate_july_case('AVI 45', 'AVI', 45),
    'AVI 60': simulate_july_case('AVI 60', 'AVI', 60),
}

overview = july_cases['AVI 0']
x = overview['state']
vc_mag = np.hypot(x[0], x[1])
figure_july_overview = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.035)
for y, name, color in [
    (vc_mag, 'v_c', '#0072bd'), (x[0], 'v_d', '#d95319'),
    (x[1], 'v_q', '#edb120'), (overview['grid_voltage'], 'v_g', '#7e2f8e'),
]:
    figure_july_overview.add_trace(go.Scatter(x=overview['time_ms'], y=y, name=name, line={'color': color}), row=1, col=1)
figure_july_overview.add_trace(go.Scatter(x=overview['time_ms'], y=x[2], name='i_ref,d'), row=2, col=1)
figure_july_overview.add_trace(go.Scatter(x=overview['time_ms'], y=x[3], name='i_ref,q'), row=2, col=1)
figure_july_overview.add_trace(go.Scatter(x=overview['time_ms'], y=x[6], name='delta', line={'color': 'black'}), row=3, col=1)
figure_july_overview.update_yaxes(title_text='Voltage (pu)', row=1, col=1)
figure_july_overview.update_yaxes(title_text='i_ref (pu)', row=2, col=1)
figure_july_overview.update_yaxes(title_text='delta (rad)', row=3, col=1)
figure_july_overview.update_xaxes(title_text='Time (ms)', row=3, col=1)
figure_july_overview.update_layout(title='July 2026 Codesim replay: AVI 0 overview', height=780, template='plotly_white', hovermode='x unified')
figure_july_overview.show(config={'displaylogo': False})

figure_july_comparison = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
colors = {'CL': '#3526a8', 'AVI 0': '#2196f3', 'AVI 45': '#65c45a', 'AVI 60': '#f1e600'}
for label in ('CL', 'AVI 0', 'AVI 45', 'AVI 60'):
    case = july_cases[label]
    state = case['state']
    figure_july_comparison.add_trace(go.Scatter(
        x=case['relative_ms'], y=np.hypot(state[0], state[1]), name=label,
        legendgroup=label, line={'color': colors[label]},
    ), row=1, col=1)
    figure_july_comparison.add_trace(go.Scatter(
        x=case['relative_ms'], y=state[6], name=label, legendgroup=label,
        showlegend=False, line={'color': colors[label]},
    ), row=2, col=1)
figure_july_comparison.update_yaxes(title_text='Voltage (pu)', row=1, col=1)
figure_july_comparison.update_yaxes(title_text='delta (rad)', row=2, col=1)
figure_july_comparison.update_xaxes(title_text='Time after recovery (ms)', row=2, col=1)
figure_july_comparison.update_layout(title='July 2026 Codesim replay: current-limiter comparison', height=650, template='plotly_white', hovermode='x unified')
figure_july_comparison.show(config={'displaylogo': False})

for label, case in july_cases.items():
    replay_peak = float(np.hypot(case['state'][0], case['state'][1]).max())
    print(f'{label:6s}: archived peak={ARCHIVED_PEAK[label]:.6f}, replay peak={replay_peak:.6f}')


CL    : archived peak=1.339704, replay peak=1.336077
AVI 0 : archived peak=1.317397, replay peak=1.324818
AVI 45: archived peak=1.334964, replay peak=1.338168
AVI 60: archived peak=1.409280, replay peak=1.411812


# First figure using `GFMSMIBAVI`

`GFMAVI` is a current device with the PCC capacitor, `Rg/Lg`, and adaptive virtual impedance embedded internally. `GFMSMIBAVI` therefore has the same one-node external structure as `VSGSMIBSystemNoCapacitor`, while retaining the capacitor dynamics required by the original MATLAB figure.


In [25]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'codesimpy').is_dir():
    ROOT = next(parent for parent in Path.cwd().resolve().parents if (parent / 'codesimpy').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from codesimpy.systems import GFMSMIBAVI

avi_parameters = GFMSMIBAVI.defaultParameters()
avi_system = GFMSMIBAVI(avi_parameters)
avi_initial = avi_system.initialize(
    ResidualTolerance=avi_parameters['residualTolerance'],
)
avi_time_s = np.unique(np.r_[
    np.arange(0, avi_parameters['validationTime_s'] + avi_parameters['outputStep_s'] / 2,
              avi_parameters['outputStep_s']),
    avi_parameters['breakpoints_s'],
])
with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
    avi_result = avi_system.simulate(
        avi_time_s, InitialState=avi_initial['state'],
        Breakpoints_s=avi_parameters['breakpoints_s'],
        saveSnapshot=False,
    )

avi_signals = avi_result['signals']
avi_clear_s = avi_result['clearingTime_s']
archived_clear_s = 1.301314011
avi_display_ms = (avi_result['time_s'] - avi_clear_s + archived_clear_s) * 1000
avi_window = (avi_display_ms >= 1298) & (avi_display_ms <= 1370)
avi_figure = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.035,
    row_heights=[0.40, 0.30, 0.30],
)
for values, name, color in [
    (avi_signals['pccVoltage'], 'v_c', '#0072bd'),
    (avi_signals['vd'], 'v_d', '#d95319'),
    (avi_signals['vq'], 'v_q', '#edb120'),
    (avi_signals['gridVoltage'], 'v_g', '#7e2f8e'),
]:
    avi_figure.add_trace(go.Scatter(
        x=avi_display_ms[avi_window], y=values[avi_window], name=name,
        line={'color': color, 'width': 2},
    ), row=1, col=1)
avi_figure.add_trace(go.Scatter(
    x=avi_display_ms[avi_window], y=avi_signals['irefDQ'][avi_window, 0],
    name='i_ref,d', line={'color': '#0072bd', 'width': 2},
), row=2, col=1)
avi_figure.add_trace(go.Scatter(
    x=avi_display_ms[avi_window], y=avi_signals['irefDQ'][avi_window, 1],
    name='i_ref,q', line={'color': '#d95319', 'width': 2},
), row=2, col=1)
avi_figure.add_trace(go.Scatter(
    x=avi_display_ms[avi_window], y=avi_signals['delta'][avi_window],
    name='delta', line={'color': 'black', 'width': 2},
), row=3, col=1)
avi_figure.update_yaxes(title_text='Voltage (pu)', row=1, col=1)
avi_figure.update_yaxes(title_text='i_ref (pu)', row=2, col=1)
avi_figure.update_yaxes(title_text='delta (rad)', row=3, col=1)
avi_figure.update_xaxes(title_text='Time (ms)', row=3, col=1)
avi_figure.update_layout(
    title='GFMSMIBAVI reproduction of the first MATLAB image',
    height=820, template='plotly_white', hovermode='x unified',
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.01},
)
post_fault = avi_result['time_s'] >= avi_clear_s
print(f'CodeSim clearing time: {avi_clear_s:.6f} s')
print(f'CodeSim PCC peak: {np.max(avi_signals["pccVoltage"][post_fault]):.6f} pu')
print(f'Maximum adaptive Rv: {np.max(avi_signals["virtualResistance"]):.6f} pu')
print('Archived MATLAB peak: 1.317397 pu')
avi_figure.show(config={'displaylogo': False})


CodeSim clearing time: 1.257441 s
CodeSim PCC peak: 1.320972 pu
Maximum adaptive Rv: 0.717049 pu
Archived MATLAB peak: 1.317397 pu
